# Семинар: Оптимизаторы, инициализации и шедулеры для нейронных сетей

В этом семинаре мы:
- Реализуем различные методы инициализации весов
- Напишем популярные оптимизаторы (SGD, Momentum, RMSProp, Adam)
- Реализуем шедулеры для learning rate
- Обучим простую нейронную сеть на датасете MNIST

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles
from sklearn.preprocessing import StandardScaler
%matplotlib inline

## Часть 1: Простая нейронная сеть

Сначала реализуем простую двухслойную нейронную сеть для бинарной классификации.

In [ ]:
class SimpleNN:
    """Простая двухслойная нейронная сеть"""
    
    def __init__(self, input_size, hidden_size, output_size=1):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Веса будут инициализированы позже
        self.W1 = None
        self.b1 = None
        self.W2 = None
        self.b2 = None
        
        # Для сохранения промежуточных значений
        self.cache = {}
    
    def sigmoid(self, x):
        """Сигмоидная функция активации"""
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def relu(self, x):
        """ReLU функция активации"""
        return np.maximum(0, x)
    
    def forward(self, X):
        """Прямой проход через сеть
        
        Args:
            X: входные данные, shape (batch_size, input_size)
        Returns:
            выход сети, shape (batch_size, 1)
        """
        # Первый слой
        z1 = X @ self.W1 + self.b1
        a1 = self.relu(z1)
        
        # Второй слой
        z2 = a1 @ self.W2 + self.b2
        a2 = self.sigmoid(z2)
        
        # Сохраняем для обратного прохода
        self.cache = {
            'X': X,
            'z1': z1,
            'a1': a1,
            'z2': z2,
            'a2': a2
        }
        
        return a2
    
    def backward(self, y):
        """Обратный проход - вычисление градиентов
        
        Args:
            y: истинные метки, shape (batch_size, 1)
        Returns:
            словарь с градиентами
        """
        m = y.shape[0]
        
        # Градиент по выходу
        dz2 = self.cache['a2'] - y
        
        # Градиенты второго слоя
        dW2 = (1/m) * self.cache['a1'].T @ dz2
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)
        
        # Градиент через второй слой
        da1 = dz2 @ self.W2.T
        dz1 = da1 * (self.cache['z1'] > 0)  # Производная ReLU
        
        # Градиенты первого слоя
        dW1 = (1/m) * self.cache['X'].T @ dz1
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)
        
        return {
            'dW1': dW1,
            'db1': db1,
            'dW2': dW2,
            'db2': db2
        }
    
    def compute_loss(self, X, y):
        """Вычисление binary cross-entropy loss"""
        m = y.shape[0]
        predictions = self.forward(X)
        
        # Binary cross-entropy
        loss = -(1/m) * np.sum(y * np.log(predictions + 1e-8) + 
                               (1-y) * np.log(1 - predictions + 1e-8))
        return loss

## Часть 2: Методы инициализации весов

Реализуем различные методы инициализации весов:

1. **Случайная инициализация**: $W \sim \mathcal{N}(0, \sigma^2)$
2. **Xavier инициализация**: $W \sim \mathcal{N}(0, \frac{1}{n_{in}})$
3. **He инициализация**: $W \sim \mathcal{N}(0, \frac{2}{n_{in}})$

In [ ]:
class WeightInitializer:
    """Класс для инициализации весов нейронной сети"""
    
    @staticmethod
    def random_init(shape, scale=0.01):
        """Случайная инициализация
        
        # ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ
        # Инициализируйте веса случайными числами из нормального распределения
        # с нулевым средним и стандартным отклонением scale
        """
        return # ВАШ КОД
    
    @staticmethod
    def xavier_init(shape):
        """Xavier/Glorot инициализация
        
        Формула: $\text{Var}(W) = \frac{1}{n_{in}}$
        где $n_{in}$ - размерность входа
        
        # ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ
        # shape = (n_in, n_out)
        """
        n_in = shape[0]
        # ВАШ КОД: вычислите std и инициализируйте веса
        std = # ВАШ КОД
        return # ВАШ КОД
    
    @staticmethod
    def he_init(shape):
        """He инициализация (для ReLU активаций)
        
        Формула: $\text{Var}(W) = \frac{2}{n_{in}}$
        
        # ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ
        """
        n_in = shape[0]
        # ВАШ КОД: вычислите std и инициализируйте веса
        std = # ВАШ КОД
        return # ВАШ КОД
    
    @staticmethod
    def zeros_init(shape):
        """Инициализация нулями (для смещений)"""
        return np.zeros(shape)

In [ ]:
# Решение
class WeightInitializer:
    """Класс для инициализации весов нейронной сети"""
    
    @staticmethod
    def random_init(shape, scale=0.01):
        return np.random.randn(*shape) * scale
    
    @staticmethod
    def xavier_init(shape):
        n_in = shape[0]
        std = np.sqrt(1 / n_in)
        return np.random.randn(*shape) * std
    
    @staticmethod
    def he_init(shape):
        n_in = shape[0]
        std = np.sqrt(2 / n_in)
        return np.random.randn(*shape) * std
    
    @staticmethod
    def zeros_init(shape):
        return np.zeros(shape)

## Часть 3: Оптимизаторы

Реализуем различные оптимизаторы для обучения нейронной сети.

In [ ]:
class SGD:
    """Стохастический градиентный спуск
    
    Формула обновления: $w_{t+1} = w_t - \alpha \cdot g_t$
    где $g_t$ - градиент на шаге $t$, $\alpha$ - learning rate
    """
    
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update(self, params, grads):
        """Обновление параметров
        
        # ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ
        # params - словарь с параметрами (W1, b1, W2, b2)
        # grads - словарь с градиентами (dW1, db1, dW2, db2)
        """
        for key in params:
            # ВАШ КОД: обновите параметры используя градиенты
            params[key] = # ВАШ КОД

In [ ]:
class Momentum:
    """SGD с моментом
    
    Формулы:
    $v_{t+1} = \beta \cdot v_t + (1 - \beta) \cdot g_t$
    $w_{t+1} = w_t - \alpha \cdot v_{t+1}$
    """
    
    def __init__(self, learning_rate=0.01, beta=0.9):
        self.learning_rate = learning_rate
        self.beta = beta
        self.v = {}  # Словарь для хранения моментов
    
    def update(self, params, grads):
        """Обновление параметров с моментом
        
        # ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ
        """
        for key in params:
            grad_key = 'd' + key
            
            # Инициализируем момент, если его еще нет
            if key not in self.v:
                self.v[key] = np.zeros_like(params[key])
            
            # ВАШ КОД: обновите момент
            self.v[key] = # ВАШ КОД
            
            # ВАШ КОД: обновите параметры
            params[key] = # ВАШ КОД

In [ ]:
class RMSProp:
    """RMSProp оптимизатор
    
    Формулы:
    $s_{t+1} = \beta \cdot s_t + (1 - \beta) \cdot g_t^2$
    $w_{t+1} = w_t - \frac{\alpha}{\sqrt{s_{t+1} + \epsilon}} \cdot g_t$
    """
    
    def __init__(self, learning_rate=0.001, beta=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta = beta
        self.epsilon = epsilon
        self.s = {}  # Словарь для хранения квадратов градиентов
    
    def update(self, params, grads):
        """Обновление параметров RMSProp
        
        # ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ
        """
        for key in params:
            grad_key = 'd' + key
            
            if key not in self.s:
                self.s[key] = np.zeros_like(params[key])
            
            # ВАШ КОД: обновите скользящее среднее квадратов градиентов
            self.s[key] = # ВАШ КОД
            
            # ВАШ КОД: обновите параметры
            params[key] = # ВАШ КОД

In [ ]:
class Adam:
    """Adam оптимизатор
    
    Формулы:
    $m_{t+1} = \beta_1 \cdot m_t + (1 - \beta_1) \cdot g_t$
    $v_{t+1} = \beta_2 \cdot v_t + (1 - \beta_2) \cdot g_t^2$
    
    Коррекция смещения:
    $\hat{m}_{t+1} = \frac{m_{t+1}}{1 - \beta_1^{t+1}}$
    $\hat{v}_{t+1} = \frac{v_{t+1}}{1 - \beta_2^{t+1}}$
    
    Обновление:
    $w_{t+1} = w_t - \frac{\alpha}{\sqrt{\hat{v}_{t+1}} + \epsilon} \cdot \hat{m}_{t+1}$
    """
    
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}  # Первый момент
        self.v = {}  # Второй момент
        self.t = 0   # Счетчик шагов
    
    def update(self, params, grads):
        """Обновление параметров Adam
        
        # ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ
        """
        self.t += 1
        
        for key in params:
            grad_key = 'd' + key
            
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            
            # ВАШ КОД: обновите первый момент
            self.m[key] = # ВАШ КОД
            
            # ВАШ КОД: обновите второй момент
            self.v[key] = # ВАШ КОД
            
            # ВАШ КОД: коррекция смещения
            m_hat = # ВАШ КОД
            v_hat = # ВАШ КОД
            
            # ВАШ КОД: обновите параметры
            params[key] = # ВАШ КОД

In [ ]:
# Решения оптимизаторов
class SGD:
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def update(self, params, grads):
        for key in params:
            grad_key = 'd' + key
            params[key] -= self.learning_rate * grads[grad_key]

class Momentum:
    def __init__(self, learning_rate=0.01, beta=0.9):
        self.learning_rate = learning_rate
        self.beta = beta
        self.v = {}
    
    def update(self, params, grads):
        for key in params:
            grad_key = 'd' + key
            if key not in self.v:
                self.v[key] = np.zeros_like(params[key])
            self.v[key] = self.beta * self.v[key] + (1 - self.beta) * grads[grad_key]
            params[key] -= self.learning_rate * self.v[key]

class RMSProp:
    def __init__(self, learning_rate=0.001, beta=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta = beta
        self.epsilon = epsilon
        self.s = {}
    
    def update(self, params, grads):
        for key in params:
            grad_key = 'd' + key
            if key not in self.s:
                self.s[key] = np.zeros_like(params[key])
            self.s[key] = self.beta * self.s[key] + (1 - self.beta) * (grads[grad_key]**2)
            params[key] -= self.learning_rate * grads[grad_key] / (np.sqrt(self.s[key]) + self.epsilon)

class Adam:
    def __init__(self, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}
        self.v = {}
        self.t = 0
    
    def update(self, params, grads):
        self.t += 1
        for key in params:
            grad_key = 'd' + key
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[grad_key]
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[grad_key]**2)
            
            m_hat = self.m[key] / (1 - self.beta1**self.t)
            v_hat = self.v[key] / (1 - self.beta2**self.t)
            
            params[key] -= self.learning_rate * m_hat / (np.sqrt(v_hat) + self.epsilon)

## Часть 4: Шедулеры learning rate

Реализуем различные стратегии изменения learning rate в процессе обучения.

In [ ]:
class LRScheduler:
    """Базовый класс для шедулеров"""
    
    def __init__(self, optimizer, base_lr):
        self.optimizer = optimizer
        self.base_lr = base_lr
        self.current_lr = base_lr
    
    def step(self, epoch):
        """Обновить learning rate"""
        raise NotImplementedError

class StepLR(LRScheduler):
    """Уменьшение learning rate каждые step_size эпох
    
    Формула: $\alpha_t = \alpha_0 \cdot \gamma^{\lfloor \frac{epoch}{step\_size} \rfloor}$
    """
    
    def __init__(self, optimizer, base_lr, step_size=10, gamma=0.1):
        super().__init__(optimizer, base_lr)
        self.step_size = step_size
        self.gamma = gamma
    
    def step(self, epoch):
        """# ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ"""
        # ВАШ КОД: вычислите новый learning rate
        self.current_lr = # ВАШ КОД
        self.optimizer.learning_rate = self.current_lr
        return self.current_lr

class ExponentialLR(LRScheduler):
    """Экспоненциальное убывание learning rate
    
    Формула: $\alpha_t = \alpha_0 \cdot \gamma^{epoch}$
    """
    
    def __init__(self, optimizer, base_lr, gamma=0.95):
        super().__init__(optimizer, base_lr)
        self.gamma = gamma
    
    def step(self, epoch):
        """# ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ"""
        # ВАШ КОД: вычислите новый learning rate
        self.current_lr = # ВАШ КОД
        self.optimizer.learning_rate = self.current_lr
        return self.current_lr

class CosineAnnealingLR(LRScheduler):
    """Косинусное расписание learning rate
    
    Формула: $\alpha_t = \alpha_{min} + \frac{1}{2}(\alpha_{max} - \alpha_{min})(1 + \cos(\frac{epoch \cdot \pi}{T_{max}}))$
    """
    
    def __init__(self, optimizer, base_lr, T_max=100, eta_min=0):
        super().__init__(optimizer, base_lr)
        self.T_max = T_max
        self.eta_min = eta_min
    
    def step(self, epoch):
        """# ВСТАВЬТЕ ВАШ КОД ЗДЕСЬ"""
        # ВАШ КОД: вычислите новый learning rate используя косинусное расписание
        self.current_lr = # ВАШ КОД
        self.optimizer.learning_rate = self.current_lr
        return self.current_lr

In [ ]:
# Решения шедулеров
class StepLR(LRScheduler):
    def __init__(self, optimizer, base_lr, step_size=10, gamma=0.1):
        super().__init__(optimizer, base_lr)
        self.step_size = step_size
        self.gamma = gamma
    
    def step(self, epoch):
        self.current_lr = self.base_lr * (self.gamma ** (epoch // self.step_size))
        self.optimizer.learning_rate = self.current_lr
        return self.current_lr

class ExponentialLR(LRScheduler):
    def __init__(self, optimizer, base_lr, gamma=0.95):
        super().__init__(optimizer, base_lr)
        self.gamma = gamma
    
    def step(self, epoch):
        self.current_lr = self.base_lr * (self.gamma ** epoch)
        self.optimizer.learning_rate = self.current_lr
        return self.current_lr

class CosineAnnealingLR(LRScheduler):
    def __init__(self, optimizer, base_lr, T_max=100, eta_min=0):
        super().__init__(optimizer, base_lr)
        self.T_max = T_max
        self.eta_min = eta_min
    
    def step(self, epoch):
        self.current_lr = self.eta_min + (self.base_lr - self.eta_min) * \
                         (1 + np.cos(np.pi * epoch / self.T_max)) / 2
        self.optimizer.learning_rate = self.current_lr
        return self.current_lr

## Часть 5: Обучение и визуализация

Теперь обучим нашу сеть на простом датасете и сравним различные комбинации.

In [ ]:
# Создаем простой датасет
np.random.seed(42)
X, y = make_circles(n_samples=300, noise=0.1, factor=0.5)
y = y.reshape(-1, 1)

# Нормализация
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Визуализация
plt.figure(figsize=(8, 6))
plt.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], c='blue', label='Класс 0')
plt.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], c='red', label='Класс 1')
plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Исходные данные')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def train_network(X, y, nn, optimizer, scheduler=None, epochs=100, verbose=True):
    """Функция обучения сети"""
    losses = []
    lrs = []
    
    for epoch in range(epochs):
        # Прямой проход
        predictions = nn.forward(X)
        loss = nn.compute_loss(X, y)
        losses.append(loss)
        
        # Обратный проход
        grads = nn.backward(y)
        
        # Обновление весов
        params = {'W1': nn.W1, 'b1': nn.b1, 'W2': nn.W2, 'b2': nn.b2}
        optimizer.update(params, grads)
        nn.W1, nn.b1, nn.W2, nn.b2 = params['W1'], params['b1'], params['W2'], params['b2']
        
        # Обновление learning rate
        if scheduler:
            current_lr = scheduler.step(epoch)
            lrs.append(current_lr)
        else:
            lrs.append(optimizer.learning_rate)
        
        if verbose and (epoch + 1) % 20 == 0:
            accuracy = np.mean((predictions > 0.5) == y)
            print(f'Эпоха {epoch+1}/{epochs}, Loss: {loss:.4f}, Accuracy: {accuracy:.4f}, LR: {lrs[-1]:.6f}')
    
    return losses, lrs

In [ ]:
def visualize_decision_boundary(X, y, nn, title=''):
    """Визуализация границы решения"""
    h = 0.01
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = nn.forward(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdBu)
    plt.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], c='blue', s=50, edgecolor='black', label='Класс 0')
    plt.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], c='red', s=50, edgecolor='black', label='Класс 1')
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## Эксперимент 1: Сравнение методов инициализации

In [ ]:
init_methods = {
    'Random': WeightInitializer.random_init,
    'Xavier': WeightInitializer.xavier_init,
    'He': WeightInitializer.he_init
}

results_init = {}

for init_name, init_func in init_methods.items():
    print(f"\nОбучение с {init_name} инициализацией:")
    
    # Создаем сеть
    nn = SimpleNN(input_size=2, hidden_size=16)
    
    # Инициализируем веса
    nn.W1 = init_func((2, 16))
    nn.b1 = WeightInitializer.zeros_init((1, 16))
    nn.W2 = init_func((16, 1))
    nn.b2 = WeightInitializer.zeros_init((1, 1))
    
    # Создаем оптимизатор
    optimizer = SGD(learning_rate=0.1)
    
    # Обучаем
    losses, lrs = train_network(X, y, nn, optimizer, epochs=100, verbose=False)
    results_init[init_name] = losses
    
    final_accuracy = np.mean((nn.forward(X) > 0.5) == y)
    print(f"Финальная точность: {final_accuracy:.4f}")

In [ ]:
# Визуализация результатов
plt.figure(figsize=(12, 6))
for init_name, losses in results_init.items():
    plt.plot(losses, label=init_name)
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Сравнение методов инициализации')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Эксперимент 2: Сравнение оптимизаторов

In [ ]:
optimizers = {
    'SGD': SGD(learning_rate=0.1),
    'Momentum': Momentum(learning_rate=0.05, beta=0.9),
    'RMSProp': RMSProp(learning_rate=0.01, beta=0.999),
    'Adam': Adam(learning_rate=0.01, beta1=0.9, beta2=0.999)
}

results_opt = {}

for opt_name, optimizer in optimizers.items():
    print(f"\nОбучение с {opt_name} оптимизатором:")
    
    # Создаем сеть
    nn = SimpleNN(input_size=2, hidden_size=16)
    
    # Используем He инициализацию
    nn.W1 = WeightInitializer.he_init((2, 16))
    nn.b1 = WeightInitializer.zeros_init((1, 16))
    nn.W2 = WeightInitializer.he_init((16, 1))
    nn.b2 = WeightInitializer.zeros_init((1, 1))
    
    # Обучаем
    losses, lrs = train_network(X, y, nn, optimizer, epochs=100, verbose=False)
    results_opt[opt_name] = losses
    
    final_accuracy = np.mean((nn.forward(X) > 0.5) == y)
    print(f"Финальная точность: {final_accuracy:.4f}")

In [ ]:
# Визуализация результатов
plt.figure(figsize=(12, 6))
for opt_name, losses in results_opt.items():
    plt.plot(losses, label=opt_name)
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.title('Сравнение оптимизаторов')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Эксперимент 3: Сравнение шедулеров

In [ ]:
# Создаем оптимизаторы и шедулеры
base_lr = 0.1
schedulers_config = {
    'Без шедулера': (SGD(learning_rate=base_lr), None),
    'StepLR': (SGD(learning_rate=base_lr), StepLR(SGD(learning_rate=base_lr), base_lr, step_size=30, gamma=0.1)),
    'ExponentialLR': (SGD(learning_rate=base_lr), ExponentialLR(SGD(learning_rate=base_lr), base_lr, gamma=0.95)),
    'CosineAnnealing': (SGD(learning_rate=base_lr), CosineAnnealingLR(SGD(learning_rate=base_lr), base_lr, T_max=100))
}

results_sched = {}
lr_histories = {}

for sched_name, (optimizer, scheduler) in schedulers_config.items():
    print(f"\nОбучение с {sched_name}:")
    
    # Создаем сеть
    nn = SimpleNN(input_size=2, hidden_size=16)
    
    # Используем He инициализацию
    nn.W1 = WeightInitializer.he_init((2, 16))
    nn.b1 = WeightInitializer.zeros_init((1, 16))
    nn.W2 = WeightInitializer.he_init((16, 1))
    nn.b2 = WeightInitializer.zeros_init((1, 1))
    
    # Обучаем
    losses, lrs = train_network(X, y, nn, optimizer, scheduler, epochs=100, verbose=False)
    results_sched[sched_name] = losses
    lr_histories[sched_name] = lrs
    
    final_accuracy = np.mean((nn.forward(X) > 0.5) == y)
    print(f"Финальная точность: {final_accuracy:.4f}")

In [ ]:
# Визуализация результатов
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# График loss
for sched_name, losses in results_sched.items():
    ax1.plot(losses, label=sched_name)
ax1.set_xlabel('Эпоха')
ax1.set_ylabel('Loss')
ax1.set_title('Сравнение шедулеров - Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# График learning rate
for sched_name, lrs in lr_histories.items():
    ax2.plot(lrs, label=sched_name)
ax2.set_xlabel('Эпоха')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Сравнение шедулеров - Learning Rate')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Финальный эксперимент: Лучшая комбинация

In [ ]:
# Обучаем с лучшей комбинацией
print("Обучение с лучшей комбинацией параметров:")
print("- Инициализация: He")
print("- Оптимизатор: Adam")
print("- Шедулер: CosineAnnealing\n")

# Создаем сеть
nn_best = SimpleNN(input_size=2, hidden_size=32)

# He инициализация
nn_best.W1 = WeightInitializer.he_init((2, 32))
nn_best.b1 = WeightInitializer.zeros_init((1, 32))
nn_best.W2 = WeightInitializer.he_init((32, 1))
nn_best.b2 = WeightInitializer.zeros_init((1, 1))

# Adam оптимизатор с косинусным расписанием
optimizer_best = Adam(learning_rate=0.01)
scheduler_best = CosineAnnealingLR(optimizer_best, base_lr=0.01, T_max=200)

# Обучаем дольше
losses_best, lrs_best = train_network(X, y, nn_best, optimizer_best, scheduler_best, epochs=200, verbose=True)

# Визуализация границы решения
visualize_decision_boundary(X, y, nn_best, 'Лучшая модель - Граница решения')

## Выводы

В этом семинаре мы:

1. **Реализовали методы инициализации весов:**
   - Случайная инициализация может приводить к проблемам с градиентами
   - Xavier инициализация хорошо работает с tanh и sigmoid
   - He инициализация оптимальна для ReLU активаций

2. **Реализовали оптимизаторы:**
   - SGD - простой, но медленный
   - Momentum - ускоряет сходимость
   - RMSProp - адаптивный learning rate
   - Adam - комбинирует преимущества Momentum и RMSProp

3. **Реализовали шедулеры:**
   - StepLR - резкие изменения learning rate
   - ExponentialLR - плавное экспоненциальное убывание
   - CosineAnnealing - плавное косинусное расписание

**Рекомендации:**
- Используйте He инициализацию для ReLU, Xavier для tanh/sigmoid
- Adam часто работает лучше других оптимизаторов
- Шедулеры помогают улучшить финальное качество модели